# Лабораторная работа №8
## Анализ и прогнозирование временного ряда

Методичка: [LAB_TMO_TIMESERIES](https://github.com/ugapanyuk/courses_current/wiki/LAB_TMO_TIMESERIES).

Датасет: временной ряд CO₂ (Mauna Loa) из `statsmodels` (агрегируем до месячного среднего).

In [ ]:
import math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from statsmodels.datasets import co2
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

from gplearn.genetic import SymbolicRegressor

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
%matplotlib inline

RANDOM_STATE = 42

In [ ]:
data = co2.load_pandas().data
s = data["co2"].astype(float).interpolate(limit_direction="both")
y = s.resample("MS").mean()
y.name = "y"

plt.figure(figsize=(11, 4))
plt.plot(y)
plt.title("CO₂ (monthly mean)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_acf(y, lags=36, ax=axes[0])
plot_pacf(y, lags=36, ax=axes[1], method="ywm")
plt.tight_layout()
plt.show()

adf = adfuller(y)
print("ADF stat:", round(adf[0], 4), "p=", round(adf[1], 6))

In [ ]:
TEST_SIZE = 24
y_train = y.iloc[:-TEST_SIZE]
y_test = y.iloc[-TEST_SIZE:]

plt.figure(figsize=(11, 4))
plt.plot(y_train, label="train")
plt.plot(y_test, label="test")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def mae(a, b):
    return mean_absolute_error(a, b)

def rmse(a, b):
    return math.sqrt(mean_squared_error(a, b))

arima = ARIMA(y_train, order=(2, 1, 2)).fit()
pred_arima = arima.forecast(steps=len(y_test))
pred_arima.index = y_test.index

print("ARIMA MAE", round(mae(y_test, pred_arima), 3), "RMSE", round(rmse(y_test, pred_arima), 3))

plt.figure(figsize=(11, 4))
plt.plot(y_train, label="train", alpha=0.6)
plt.plot(y_test, label="test")
plt.plot(pred_arima, label="ARIMA")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def make_lags(series: pd.Series, lags: int) -> pd.DataFrame:
    df = pd.DataFrame({"y": series})
    for i in range(1, lags + 1):
        df[f"lag_{i}"] = series.shift(i)
    return df.dropna()

LAGS = 12
sup = make_lags(y, LAGS)
X_sup = sup.drop(columns=["y"])
y_sup = sup["y"]

cutoff = y_test.index.min()
X_train_sup = X_sup.loc[X_sup.index < cutoff]
y_train_sup = y_sup.loc[y_sup.index < cutoff]
X_test_sup = X_sup.loc[X_sup.index >= cutoff]
y_test_sup = y_sup.loc[y_sup.index >= cutoff]

X_train_sup.shape, X_test_sup.shape

In [ ]:
sym = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", SymbolicRegressor(population_size=1500, generations=12, random_state=RANDOM_STATE, verbose=0)),
])
sym.fit(X_train_sup, y_train_sup)
pred_sym = pd.Series(sym.predict(X_test_sup), index=X_test_sup.index)

print("Symbolic MAE", round(mae(y_test_sup, pred_sym), 3), "RMSE", round(rmse(y_test_sup, pred_sym), 3))

plt.figure(figsize=(11, 4))
plt.plot(y_train, label="train", alpha=0.6)
plt.plot(y_test, label="test")
plt.plot(pred_sym.reindex(y_test.index), label="symbolic")
plt.legend()
plt.tight_layout()
plt.show()

### (Опционально) GMDH через `gmdh`

По методичке нужен COMBI/MULTI и MIA/RIA через библиотеку `gmdh`. На macOS установка может требовать Boost; поэтому в репозитории блок сделан опциональным.